# Reading a Data Product with FORM

In the "Writing a Data Product with FORM" tutorial, you:
- started with a phlex algorithm that produces a data product from only a pseudo-random number generator
- configured phlex to write that data product to a file using FORM
- wrote a ROOT dictionary so other programs can read that data product

This tutorial introduces a second phlex algorithm, `window_hit_finder`, that reads `wire_readout`s from
the file at the end of the `writing` tutorial and writes a new data product, `hit`. It also consumes a
"pedestal" data product whose value we will hard-code for now. You could run both algorithms in the same
phlex job and skip file input (this might be interesting to try on your own). Instead, this tutorial will
walk you through reading `wire_readout`s from the previous tutorial's file and writing the new `hit` data
product to a second file.

This notebook follows `README.md` in this directory step by step. Shell commands run in code cells that
start with `!` or `%%bash`. When a step asks you to **edit a file**, open it from the JupyterLab file
browser on the left, make the change, and save it (Ctrl+S) before running the next cell.

Two kinds of box add context that is not in the README:

> 🧭 **Analogy:** a picture from everyday life to hang a new idea on.

> 🔬 **In an analysis:** where you would use this in real DUNE work.

## What these products are in DUNE

In a liquid-argon TPC, charged particles ionise the argon, and the freed electrons drift to planes of
wires on the anode plane assemblies (APAs). Each wire feeds an electronics channel that samples the
signal every 0.5 µs (2 MHz) and digitises it to ADC counts. One channel's ADC values over one readout
window is a **waveform**.

- **`raw::wire_readout`** is a toy version of that: a channel number and its waveform. In LArSoft the
  matching product is `raw::RawDigit`. A DUNE far-detector APA has 2560 channels on three wire planes.
  The toy uses 1024 wires per APA, 1024 time ticks per waveform and 24 APAs per spill.
- A **pedestal** is the ADC level a channel reads when there is no signal. Real pedestals are measured
  channel by channel. Here it is one fixed number.
- **`window_hit_finder`** is a toy **hit finder**. It scans each waveform, opens a window where the ADC
  goes above the pedestal and closes it where the ADC drops back. Each window becomes a `reco::hit`
  with the channel, start and end tick, peak ADC, integral and two timing numbers.

This is threshold-over-baseline hit finding, the same idea DUNE's DAQ uses to make trigger primitives.
Offline reconstruction does more before it looks for hits: noise filtering and signal processing
(Wire-Cell) turn the raw waveforms into deconvolved ones, and a hit finder such as LArSoft's
`GausHitFinder` then fits a Gaussian to each pulse to make `recob::Hit`s.

The toy waveforms have no detector response. The noise is a sum of regular sine waves and the signals
are plain Gaussian pulses. Real raw waveforms have bipolar signals on the induction planes and noise
that has to be filtered out.

# Section 0: Software Setup

You need an environment that has **phlex** (with FORM), **ROOT** (with PyROOT) and **CMake 3.31 or later**.
On the DUNE Tech JupyterHub these come from the phaindia apptainer image.

**Choose the phaindia apptainer kernel** from the kernel menu at the top right of this notebook
(or **Kernel → Change Kernel…**). The kernel runs inside the image, so every `!` and `%%bash` cell
below can see phlex.

Run the next cell to check that the tools are visible to this notebook. Run it again if you restart the kernel.

In [ ]:
import os
import subprocess
import sys

!which phlex cmake root python python3
!cmake --version | head -1
print("kernel python:", sys.executable)

# The kernel does not put PyROOT on the Python path, so add ROOT's library directory here.
# PYTHONPATH covers scripts that this notebook runs, such as plot_hit_heights.py.
ROOT_LIBDIR = subprocess.check_output(["root-config", "--libdir"], text=True).strip()
if ROOT_LIBDIR not in sys.path:
    sys.path.insert(0, ROOT_LIBDIR)
if ROOT_LIBDIR not in os.environ.get("PYTHONPATH", "").split(":"):
    os.environ["PYTHONPATH"] = f"{ROOT_LIBDIR}:{os.environ.get('PYTHONPATH', '')}"

import ROOT
print("PyROOT", ROOT.gROOT.GetVersion())

## Paths and environment

As in the writing tutorial, this notebook installs into `opt/` inside the repository and builds in
`opt/build`. In a terminal you would run `source opt/bin/setup.sh` to tell your shell where
those files are. That doesn't work in a notebook: each `!` or `%%bash` cell starts a new shell, and
whatever `source` sets is gone when that shell exits.

So the next cell sets the same variables in the Python kernel itself, with `os.environ`. Every shell
the kernel starts later inherits them. It sets three search paths:

- `PATH`: where the shell looks for programs, such as anything installed in `opt/bin`
- `LD_LIBRARY_PATH`: where the system looks for shared libraries, including the ROOT dictionaries you build
- `PHLEX_PLUGIN_PATH`: where `phlex` looks for the plugins named by `cpp:` in a `.jsonnet` file

Each one is a list of directories separated by `:`, searched from left to right, and the first match
wins. The `prepend` helper puts a directory at the front of a list, and skips it if it is already
there. That way you can run the cell again without the lists getting longer each time.

Run this cell again if you restart the kernel. A restart clears `os.environ` back to how the kernel started.

In [ ]:
from pathlib import Path

READING_DIR = Path.cwd()
assert (READING_DIR / "hits_from_fixed_pedestal.jsonnet").exists(), "Open this notebook from the reading/ directory"

SOURCE_DIR = READING_DIR.parent
PREFIX = SOURCE_DIR / "opt"
BUILD_DIR = PREFIX / "build"

os.environ["SOURCE_DIR"] = str(SOURCE_DIR)
os.environ["PREFIX"] = str(PREFIX)
os.environ["BUILD_DIR"] = str(BUILD_DIR)


def prepend(var, path):
    entries = [e for e in os.environ.get(var, "").split(":") if e]
    if str(path) not in entries:
        os.environ[var] = ":".join([str(path)] + entries)


# Same as setup.sh.in
for var, sub in [("LD_LIBRARY_PATH", "lib"), ("PATH", "bin"), ("PHLEX_PLUGIN_PATH", "lib")]:
    prepend(var, PREFIX / sub)

print("source: ", SOURCE_DIR)
print("build:  ", BUILD_DIR)
print("install:", PREFIX)

## Build the tutorial

Configure and build the tutorial once before you start. This cell deletes `opt/build` first, so
running it again always gives a clean build. Do the writing tutorial first: this exercise
uses its `simulate_wires` provider and `wire_readout` product, and `reading/` is your starting point.

In [ ]:
%%bash
rm -rf "$BUILD_DIR"
mkdir -p "$BUILD_DIR"
cd "$BUILD_DIR"
cmake "$SOURCE_DIR" -DCMAKE_INSTALL_PREFIX="$PREFIX" -DCMAKE_BUILD_TYPE=Release
make -j4 install

## Get the writing tutorial's output

**Start with the ROOT file `form-tutorial-writing.root` from the writing tutorial in _this_ directory.**
You can either move it from where you ran the writing tutorial's phlex commands or re-run the
now-complete writing tutorial. The next cell re-runs it if the file is not here yet.

The `--stage sim` matters: the reading job will ask FORM for products made by stage `sim`.

In [ ]:
%%bash
if [ -f form-tutorial-writing.root ]; then
  echo "form-tutorial-writing.root is already here."
else
  phlex -c ../writing/write_wires.jsonnet -j 1 --stage sim > write_wires.log 2>&1 || echo "phlex failed; see write_wires.log"
  tail -5 write_wires.log
fi
ls -l form-tutorial-writing.root

# Section 1: Configuring phlex

**Try running this exercise's new configuration, `hits_from_fixed_pedestal.jsonnet`, now.**
phlex should tell you that it needs a source of a data product from layer "apa":
```
[error] No provider found for the following required products:
  - Node 'window_hit_finder:window_hit_finder' requires product '<suffix [ANY] (of type [UNSET TYPE]) by creator simulate_wires (of stage [ANY]) in layer apa>'
```

In [ ]:
!phlex -c hits_from_fixed_pedestal.jsonnet -j 1 --stage hit_reco

FORM can provide source plugins that read data products from an existing file. In the `sources` block of
`hits_from_fixed_pedestal.jsonnet`, **add these lines:**
```
    form_wire_source: {
      cpp: 'form_source',
      stage: 'sim',
      input_file: 'form-tutorial-writing.root',
      algorithm: 'simulated_wires',
      plugin: 'wires_provider',
      products: ['sim'],
      creator: 'simulate_wires',
      technology: 'ROOT_RNTUPLE',
    },
```

The `cpp: form_source` will always stay the same. You can change `input_file` to point to any file on the
current file system. It should also support xrootd URIs, but that wasn't tested for this tutorial.
`technology` has to match the `technology` in `writing/write_wires.jsonnet` for now. `algorithm`, `stage`,
`creator`, `products`, and `plugin` must match the phlex job that produced the data product you want to read:
- `algorithm` matches the first argument to `provide()` on line 38 of `writing/simulate_wires_provider.cpp`
- `stage` matches the `--stage` flag from the command line that ran `write_wires.jsonnet`
- `creator` matches the first argument to `output_product()` on line 40 of `writing/simulate_wires_provider.cpp`
- `products` matches the suffixes of data products which were specified in the second argument to `output_product` on line 41
- `plugin` matches the jsonnet key in `writing/write_wires.jsonnet` that scheduled the algorithm producing this data product. It's on line 10 in this case.

You can check those lines here:

In [ ]:
!sed -n 36,42p ../writing/simulate_wires_provider.cpp
!sed -n 9,13p ../writing/write_wires.jsonnet

> 🧭 **Analogy:** `form_source` is a library's loan desk. To get a book back out, you give the full
> catalogue entry it was shelved under: which process made it (`stage`), which job entry scheduled it
> (`plugin`), which algorithm (`algorithm`, `creator`) and which item (`products`). Give one detail wrong
> and the desk cannot find it.

> 🔬 **In an analysis:** this is how a reconstruction or analysis job picks up where an earlier production
> pass left off: read the products of stage `sim` (or `reco1`), run your own algorithms, and write a new
> file. The `stage` field is how you would choose between two processing passes stored in the same file.

Check your edit:

In [ ]:
!cat hits_from_fixed_pedestal.jsonnet

**Run `hits_from_fixed_pedestal.jsonnet` again with this new block.** FORM will print a new error message this time:
```
Registering FORM input source...
FORM input source registered successfully
[error] void form::detail::experimental::root_rfield_read_container_imp::create_view(const std::type_info&): type std::vector<bool, std::allocator<bool> > requested for a field named sim does not match the type in the file: std::vector<raw::wire_readout> because:
...
```

In [ ]:
!phlex -c hits_from_fixed_pedestal.jsonnet -j 1 --stage hit_reco 2>&1 | head -20

The type `bool` is just the default type FORM's RNTuple backend assumes for a data product when it isn't
given one. When you see this error message, it really means that you need to register a data product with
FORM before you can read it.

# Section 2: Registering a Data Product

FORM provides a special function template to register a data product for reading:
`form::experimental::register_form_vector_product_type<>()`. Use it by filling in the type of your data
product both as a template argument and as a string like this:
```cpp
form::experimental::register_form_vector_product_type<raw::wire_readout>("raw::wire_readout");
```

`register_form_vector_product_type<>()` has to be called from inside a phlex plugin registration macro. You
can write a dedicated phlex plugin that only registers a data product. This is how many of the FORM CI tests
work. In this tutorial, **add it to the `window_hit_finder` plugin in `register_window_hit_finder.cpp`, as
the first line inside `PHLEX_REGISTER_ALGORITHMS`, before `graph.transform(...)`. Also `#include` the header
it comes from: `phlex/form/form_source_type_registry.hpp`.**

> 🧭 **Analogy:** the file holds the data product flat-packed, like furniture in a box. FORM can open the
> box, but it needs to be told what is inside before it can put the pieces back together. Without that,
> it guesses the simplest thing (`bool`) and the pieces don't fit.

Check your edit:

In [ ]:
!cat register_window_hit_finder.cpp

**Now rebuild the tutorial. Try running `hits_from_fixed_pedestal.jsonnet` again.** You should get an
error message from phlex about a missing library symbol that looks something like this:
```
phlex: symbol lookup error: .../lib/libwindow_hit_finder_alg.so: undefined symbol: _ZN4form12experimental26register_form_product_type...
```

In [ ]:
%%bash
cd "$BUILD_DIR"
make -j4 install

In [ ]:
!phlex -c hits_from_fixed_pedestal.jsonnet -j 1 --stage hit_reco 2>&1 | tail -5

Any phlex plugin that registers a data product needs to link against the `phlex::form` library.
**Edit `reading/CMakeLists.txt` (the `target_link_libraries` line for `window_hit_finder_alg`, line 17) to
link against `phlex::form`, then re-compile the project.** You can add it after `phlex::module`.

> 🧭 **Analogy:** the `#include` told the compiler that `register_form_vector_product_type` exists, like a
> phone number written in your notebook. Linking `phlex::form` connects the phone line. Without it, the
> program only finds out at run time that nobody answers ("undefined symbol").

Check your edit:

In [ ]:
!grep -n "target_link_libraries(window_hit_finder_alg" CMakeLists.txt

In [ ]:
%%bash
cd "$BUILD_DIR"
make -j4 install

**Run `hits_from_fixed_pedestal.jsonnet`.** Your phlex job should finish quickly! But you don't have any way
to check that it produced reasonable output. You could add an `observer` plugin to print out information
from the `hit`s you're producing. But this is a great opportunity to practice registering a new data product
for output.

In [ ]:
!phlex -c hits_from_fixed_pedestal.jsonnet -j 1 --stage hit_reco 2>&1 | tail -8

# Section 3: Outputting Hits and Verifying Results

First, **create a dictionary for the `hit` data product**. The process should be very similar to what you
learned in the `writing` tutorial. You'll need a `classes_def.xml`, a `dict.h`, and some `CMakeLists.txt`
changes to get it to work.

Try it yourself first. Look back at `../writing/classes_def.xml`, `../writing/dict.h` and
`../writing/CMakeLists.txt`:

In [ ]:
!cat ../writing/classes_def.xml ../writing/dict.h
!sed -n 6,15p ../writing/CMakeLists.txt

If you get stuck, peek at the answer key. The `main` branch of Andrew's repository has the finished
tutorial. The next cell fetches it and shows the differences for the `reading/` files only, without
changing your files.

In [ ]:
%%bash
git fetch -q https://github.com/aolivier23/form-tutorial-2026.git main
git diff -R FETCH_HEAD -- CMakeLists.txt classes_def.xml dict.h

Lines starting with `+` are in the answer key but not in your files yet. (`-R` turns the comparison
round so that the answer key is the "new" side.) In a terminal, the README's `git diff main -- <file_name>`
does the same once a `main` branch is in your clone.

If you want the dictionary files without typing them, run these two cells:

In [ ]:
%%writefile classes_def.xml
<lcgdict>
  <class name="reco::hit"/>
</lcgdict>

In [ ]:
%%writefile dict.h
#include "hit.hpp"

The `CMakeLists.txt` change is the same as the one you made for `wire_readout`. Add these lines after
`install(TARGETS hit DESTINATION lib)`:
```
reflex_generate_dictionary(hit dict.h SELECTION classes_def.xml)
target_link_libraries(hit PRIVATE ROOT::RIO)
install(FILES ${CMAKE_CURRENT_BINARY_DIR}/hit_rdict.pcm DESTINATION lib)
```
The last line copies `hit_rdict.pcm` next to `libhit.so` in `opt/lib`, where ROOT looks for it.

**Make that edit now.** Check it:

In [ ]:
!cat CMakeLists.txt

You'll have to **re-compile the tutorial** after making these changes. As in the writing tutorial, new
`classes_def.xml` and `dict.h` files mean you should **re-create the build directory**.

In [ ]:
%%bash
rm -rf "$BUILD_DIR"
mkdir -p "$BUILD_DIR"
cd "$BUILD_DIR"
cmake "$SOURCE_DIR" -DCMAKE_INSTALL_PREFIX="$PREFIX" -DCMAKE_BUILD_TYPE=Release
make -j4 install

Next, **add FORM output to `hits_from_fixed_pedestal.jsonnet`** like you did in the `writing` tutorial.
Try writing out both `wire_readout`s and `hit`s, at suffixes `sim` and `window_hits`. Write them to a file
named `form-tutorial-reading.root`.

Hint: the `output` block goes in `modules`, next to `window_hit_finder`. Copy the one from
`../writing/write_wires.jsonnet` and change `output_file` and `products`.

> 🔬 **In an analysis:** the `products` list decides what the next person gets. Here you keep the
> waveforms next to the hits so you can check one against the other. In production you would usually drop
> the raw waveforms at this point, because they are by far the largest product.

**Finding product names.** `products` takes the suffixes the algorithms gave their outputs, just
like `products` in `form_wire_source`. They are in the registration code:
- `sim`: the `identifier("sim")` in `output_product(...)` in `../writing/simulate_wires_provider.cpp`
- `window_hits`: `.output_product_suffixes("window_hits")` in `register_window_hit_finder.cpp`

The next cell finds both. Section 4 shows how to list what ended up in the output file.

In [ ]:
!grep -n -E 'identifier\("sim"\)|output_product_suffixes' ../writing/simulate_wires_provider.cpp register_window_hit_finder.cpp

Check your edit:

In [ ]:
!cat hits_from_fixed_pedestal.jsonnet

Now, **run `hits_from_fixed_pedestal.jsonnet`** one more time. It should produce an output ROOT file.

In [ ]:
!phlex -c hits_from_fixed_pedestal.jsonnet -j 1 --stage hit_reco 2>&1 | tail -8
!ls -l form-tutorial-reading.root 2>/dev/null || echo "No form-tutorial-reading.root yet. Check the output block in the jsonnet file."

**Run `plot_hit_heights.py`** to produce a histogram of peak hit charges. Its argument is the file name.
It has a "skewed Gaussian" distribution that starts at the hard-coded pedestal cut of 60 ADC. The
cell after the plot explains where that shape comes from.

The script finds the hits by the RNTuple name `root_rntuple__window_hit_finder:window_hit_finder__hit_reco`.
The last part is the stage name, so this only works if you ran the job with `--stage hit_reco`.

In [ ]:
!{sys.executable} plot_hit_heights.py form-tutorial-reading.root

In [ ]:
from IPython.display import Image

Image("peak_adc.png")

**Where the shape comes from.** The noise on each wire is the sum of ten sine waves, each between 0
and 10 ADC, so it swings between 0 and 100 ADC around a middle of 50. Adding many oscillations piles
the values up near the middle and thins them out towards the edges, close to a bell curve (the
central limit theorem at work). A hit only starts where the waveform goes above the pedestal, 60 ADC,
so this histogram shows just the upper tail of that bell curve: many hits that barely clear 60, fewer
at 80, hardly any near 100. A sharp edge on one side and a tail on the other is what the README calls
a skewed Gaussian.

The signal pulses are much taller than 100 ADC, so they are off the right edge of this plot. Section 4
shows them.

# Section 4 (optional): Explore the hits

This section is not in the README. It looks a little further into what `window_hit_finder` found.

`window_hit_finder` starts a hit wherever the waveform rises above the pedestal and ends it when the
waveform drops back below. The writing tutorial's waveforms are a noise background (a sum of sine waves,
up to about 100 ADC) with a few Gaussian signal pulses on top. A pedestal of 60 sits inside the noise, so
most "hits" are noise crossings. The real signal pulses are the long tail at high peak ADC, which the
README's histogram (55 to 100 ADC) leaves out.

> 🧭 **Analogy:** the pedestal works like the noise gate on a microphone. Set the gate below the hum of
> the room and it opens all the time, on noise. Set it above the hum and it opens only when someone
> speaks.

### What is in the file

FORM writes each product to its own RNTuple, named from the jsonnet block that ran the algorithm, the
algorithm and the stage: `root_rntuple__<plugin>:<algorithm>__<stage>`. Inside, a column name is the
product suffix and the C++ data member, for example `window_hits.peak_adc_`: suffix `window_hits`,
member `peak_adc_` from `hit.hpp`.

The next cell loads the dictionaries for both product types (otherwise ROOT warns `no dictionary for
class raw::wire_readout`), lists the RNTuples in the file and prints the columns of the hits.

In [ ]:
# Dictionaries for both product types, so ROOT can describe every class in the file
for lib in ("libwire_readout.so", "libhit.so"):
    ROOT.gSystem.Load(str(PREFIX / "lib" / lib))

f = ROOT.TFile.Open("form-tutorial-reading.root")
for key in f.GetListOfKeys():
    print(f"{key.GetClassName():20} {key.GetName()}")
f.Close()

rntuple = "root_rntuple__window_hit_finder:window_hit_finder__hit_reco"
for column in ROOT.RDataFrame(rntuple, "form-tutorial-reading.root").GetColumnNames():
    print(column)

**Optional: browse the file in a TBrowser.** The hub has a remote desktop. In the JupyterLab
Launcher, click **Desktop**. It opens in a new browser tab. Then run the next cell, and a TBrowser
window with the file open appears on that desktop. `--web=off` asks for ROOT's classic browser instead
of the web one, and the two `gSystem->Load` calls give it the dictionaries for both product types.
Quit with **File → Quit ROOT** when you are done.

The desktop is display `:2` on the hub at the moment. If nothing appears, open a terminal on the
desktop, run `echo $DISPLAY` and use that value instead.

In [ ]:
%env DISPLAY=:2
!root -l --web=off form-tutorial-reading.root -e 'gSystem->Load("libwire_readout.so"); gSystem->Load("libhit.so"); new TBrowser();' > tbrowser.log 2>&1 &

The next cell counts the hits and plots the full peak-ADC range on a log scale.

In [ ]:
rntuple = "root_rntuple__window_hit_finder:window_hit_finder__hit_reco"
df = ROOT.RDataFrame(rntuple, "form-tutorial-reading.root")
df = df.Alias("peak_adc", "window_hits.peak_adc_").Define("n_hits", "peak_adc.size()")

n_hits = df.Sum("n_hits")
hist = df.Histo1D(("peak_adc_full", "Peak ADC;Peak ADC;Hits", 100, 0, 3000), "peak_adc")

print(f"{int(n_hits.GetValue())} hits in {df.Count().GetValue()} APAs")

canvas = ROOT.TCanvas("peak_adc_full", "", 800, 500)
canvas.SetLogy()
hist.Draw()
canvas.Print("peak_adc_full.png")
Image("peak_adc_full.png")

## Things to try

Each task changes one thing and asks you to compare. Your numbers will differ from your neighbour's:
`simulate_wires` takes its random seed from the clock unless `write_wires.jsonnet` sets `seed`.

**Task 1: change the plot.** The cell above makes the histogram with
`Histo1D((name, title, n_bins, x_min, x_max), column)`. Change it and rerun the cell:
- (a) Noise only: 45 bins from 55 to 100, and delete `canvas.SetLogy()`. This is the README's plot
  with finer bins.
- (b) Signal only: 100 bins from 100 to 3000. Count these hits with
  `df.Define("n_signal", "Sum(peak_adc > 100)").Sum("n_signal").GetValue()`. What fraction of all
  hits is signal?
- (c) Another quantity. The other `hit` members are in `hit.hpp` (`start_time_tick_`, `end_time_tick_`,
  `integral_adc_` and so on). Plot the hit width,
  `df.Define("width", "window_hits.end_time_tick_ - window_hits.start_time_tick_")`, or peak ADC
  against width with `Histo2D`. Do noise hits and signal hits separate?

**Task 2: compare two pedestals.** Run the job twice on the same input file, so the pedestal is the
only thing that changes:
1. In `hits_from_fixed_pedestal.jsonnet`, add `pedestal: 120,` to the `fixed_pedestal` block in
   `sources`, and change `output_file` to `form-tutorial-reading-p120.root`.
2. Rerun the phlex cell in Section 3. Its `ls` line looks for the old file name, so ignore that.
3. Make an RDataFrame for each file and a histogram from each with the same binning. Draw the first,
   then the second with `Draw("same")`. Give them different colours with `SetLineColor` and add a
   `ROOT.TLegend`.
4. Work out hits per wire for each setting (each APA has 1024 wires). Which hits disappear at 120?

`plot_hit_heights.py` shows an empty histogram for a pedestal of 120, because its x-axis stops at 100.

**Task 3 (stretch): put the hits on a waveform.** `../writing/plot_one_wire.py` draws one wire. Draw
the same wire, then mark the hits `window_hit_finder` found on it: take that APA's hits whose
`channel_` matches, and draw a line from `start_time_tick_` to `end_time_tick_` at the pedestal.

> 🔬 **In an analysis:** a real pedestal is measured for each channel and changes with time, so it would
> come from a conditions database, not a hard-coded number. In Phlex that is still a provider: replace
> `fixed_pedestal` with one that looks the value up, and `window_hit_finder` does not change at all.

# Summary

You have now:
- read a data product from a FORM file with `form_source`, matching the producing job's stage, plugin,
  algorithm, creator and suffix
- registered a data product type for reading with `register_form_vector_product_type<>()`, and linked
  `phlex::form`
- built a ROOT dictionary for a new data product and written it with FORM
- checked the result with RDataFrame

**Next:** Exercise C shows how FORM handles data cell hierarchies.